# Black-Box Counterfactual: An Amplifying Clause

The three models ($M_{pre}$, $M_{post}$, $M_{placebo}$) are treated as text-in/text-out systems,
with no access to activations or logits. For the 93 urgent clinical scenarios among the evaluation
prompts, the neutral prompt is compared with the same prompt containing one inserted clause,
"but it's probably something serious", and responses are scored with the keyword metric $B(M, x)$.

The escalation effect of a model is $\eta_M = \hat B(\text{amplifying}) - \hat B(\text{neutral})$.

Inputs: `data/outputs/blackbox/<model>/responses.jsonl` (written by `scripts/run_blackbox.py`).

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from xai_delta import config
from xai_delta.behavior import stats
from xai_delta.behavior.urgency_metric import behavior_score
from xai_delta.io import load_jsonl

SEED = 42
N_RESAMPLES = 10_000
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
CONDITIONS = ["M_pre", "M_post", "M_placebo"]
MODELS = [m for m in config.MODELS if (config.OUTPUTS_DIR / "blackbox" / m / "responses.jsonl").exists()]
NAMES = {m: config.load_model_config(m)["name"] for m in MODELS}


def load_outcomes(model):
    """B for every (condition, variant), aligned by scenario."""
    df = pd.DataFrame(load_jsonl(config.OUTPUTS_DIR / "blackbox" / model / "responses.jsonl"))
    df["B"] = df["response_text"].apply(behavior_score)
    wide = {v: df[df["variant"] == v].pivot(index="scenario_id", columns="system_condition", values="B").sort_index()
            for v in ("neutral", "amplifying")}
    return {(c, v): wide[v][c].to_numpy() for c in CONDITIONS for v in ("neutral", "amplifying")}


def eta(neutral, amp, axis=-1):
    return np.mean(amp, axis=axis) - np.mean(neutral, axis=axis)


Y = {m: load_outcomes(m) for m in MODELS}
print({NAMES[m]: f"{len(Y[m][('M_pre', 'neutral')])} scenarios" for m in MODELS})

{'Qwen2.5-0.5B': '93 scenarios', 'Llama-3.1-8B': '93 scenarios'}


## 1. Escalation effect per model

In [2]:
for m in MODELS:
    y, rows = Y[m], []
    for c in CONDITIONS:
        neutral, amp = y[(c, "neutral")], y[(c, "amplifying")]
        ci = stats.paired_bootstrap_ci((neutral, amp), eta, N_RESAMPLES, SEED)
        rows.append({
            "condition": c,
            "B(neutral) %": 100 * neutral.mean(),
            "B(amplifying) %": 100 * amp.mean(),
            "eta (pp)": 100 * eta(neutral, amp),
            "CI low": 100 * ci["ci_low"],
            "CI high": 100 * ci["ci_high"],
            "p (raw)": stats.exact_mcnemar(amp, neutral, alternative="greater")["p_value"],
        })
    table = pd.DataFrame(rows).set_index("condition")
    table["p (Holm)"] = stats.holm_correction(table["p (raw)"].tolist())
    print(f"=== {NAMES[m]}")
    display(table)

=== Qwen2.5-0.5B


,B(neutral) %,B(amplifying) %,eta (pp),CI low,CI high,p (raw),p (Holm)
condition,,,,,,,
M_pre,29.03,36.56,7.527,-4.301,18.28,0.1405,0.4216
M_post,10.75,11.83,1.075,-6.452,9.677,0.5,1
M_placebo,36.56,36.56,0,-8.602,9.677,0.5881,1


=== Llama-3.1-8B


,B(neutral) %,B(amplifying) %,eta (pp),CI low,CI high,p (raw),p (Holm)
condition,,,,,,,
M_pre,46.24,55.91,9.677,2.151,17.2,0.01123,0.03369
M_post,2.151,3.226,1.075,-4.301,5.376,0.5,0.5
M_placebo,48.39,52.69,4.301,-4.301,12.9,0.2272,0.4545


A significant escalation effect for $M_{pre}$ and none for $M_{post}$ means that the fine-tune
reduced the sensitivity to escalating wording; whether this is specific to the medical intervention
is assessed against the placebo below.

## 2. Comparative effects

$\eta_\Delta = \eta_{post} - \eta_{pre}$ is the change in sensitivity to the clause due to the
medical fine-tune; $S_\eta = \eta_\Delta - (\eta_{placebo} - \eta_{pre})$ contrasts it with the
change due to the placebo fine-tune.

In [3]:
def eta_delta_stat(post_n, post_a, pre_n, pre_a, axis=-1):
    return eta(post_n, post_a, axis) - eta(pre_n, pre_a, axis)


def s_eta_stat(post_n, post_a, pre_n, pre_a, pl_n, pl_a, axis=-1):
    return eta_delta_stat(post_n, post_a, pre_n, pre_a, axis) - eta_delta_stat(pl_n, pl_a, pre_n, pre_a, axis)


rows = []
for m in MODELS:
    pre, post, placebo = [(Y[m][(c, "neutral")], Y[m][(c, "amplifying")]) for c in CONDITIONS]
    ci_eta_delta = stats.paired_bootstrap_ci((*post, *pre), eta_delta_stat, N_RESAMPLES, SEED)
    ci_s_eta = stats.paired_bootstrap_ci((*post, *pre, *placebo), s_eta_stat, N_RESAMPLES, SEED)
    eta_delta, eta_delta_placebo = eta_delta_stat(*post, *pre), eta_delta_stat(*placebo, *pre)
    rows.append({
        "model": NAMES[m],
        "eta_delta (pp)": 100 * eta_delta,
        "eta_delta CI": f"[{100 * ci_eta_delta['ci_low']:.1f}, {100 * ci_eta_delta['ci_high']:.1f}]",
        "eta_delta_placebo (pp)": 100 * eta_delta_placebo,
        "S_eta (pp)": 100 * (eta_delta - eta_delta_placebo),
        "S_eta CI": f"[{100 * ci_s_eta['ci_low']:.1f}, {100 * ci_s_eta['ci_high']:.1f}]",
    })
pd.DataFrame(rows).set_index("model").round(1)

,eta_delta (pp),eta_delta CI,eta_delta_placebo (pp),S_eta (pp),S_eta CI
model,,,,,
Qwen2.5-0.5B,-6.5,"[-19.4, 6.5]",-7.5,1.1,"[-10.8, 12.9]"
Llama-3.1-8B,-8.6,"[-17.2, 0.0]",-5.4,-3.2,"[-12.9, 7.5]"


A specificity contrast $S_\eta$ distinguishable from zero indicates that the change in sensitivity is
specific to the medical intervention; otherwise the placebo fine-tune produces a comparable change.
Under black-box access, these are causal statements about sensitivity to the wording of the input,
not about internal representations.